
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/04_functions/04_functions.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 04 — Functions

**Learning Objectives:** def, return, args, *args/**kwargs, scope, lambda, decorators

---

## 4.1 Defining and Calling Functions

**What is a function?**
A function is a named, reusable block of code. You define it once and call it as many times as you need. Functions are the primary way to organise code and avoid repetition.

**Why use functions?**
- **DRY principle:** Don't Repeat Yourself. Write the logic once, use it everywhere.
- **Readability:** Well-named functions make code self-documenting
- **Testing:** Small functions with clear inputs/outputs are easy to test
- **Separation of concerns:** Each function should do ONE thing well

**Anatomy of a function:**
```python
def function_name(parameter1, parameter2):
    # docstring explaining what the function does
    # body: code that runs when function is called
    return result   # optional, but usually present
```

In [ ]:
def greet(name):
    """
    Return a personalised greeting.
    
    Args:
        name (str): The person's name
    Returns:
        str: A greeting string
    """
    return f"Hello, {name}! Welcome to Python."

# Calling the function
message = greet("Alice")
print(message)

# Functions are values — you can inspect them
print(greet.__name__)       # function name
print(greet.__doc__)        # the docstring

# Function without return value implicitly returns None
def say_hi(name):
    print(f"Hi, {name}!")

result = say_hi("Bob")
print(result)   # None

In [ ]:
# Default parameter values
# If the caller does not provide the argument, the default is used
def power(base, exponent=2):
    """Raise base to the power of exponent (default: squared)."""
    return base ** exponent

print(power(3))       # 9  — uses default exponent=2
print(power(3, 3))    # 27 — caller provides exponent
print(power(2, 10))   # 1024

# Keyword arguments — you can specify arguments by name
print(power(exponent=3, base=2))   # order doesn't matter with keywords
print(power(base=5))               # uses default for exponent

## 4.2 Multiple Return Values

**How Python returns multiple values:**
Python functions can return multiple values by separating them with commas. Python bundles them into a tuple, and the caller can unpack them.

This is extremely common in data science — functions that return a model AND its score, or the training AND testing data after a split.

In [ ]:
def analyse(numbers):
    """
    Compute basic statistics for a list of numbers.
    
    Returns: mean, minimum, maximum (as a tuple)
    """
    mean = sum(numbers) / len(numbers)
    minimum = min(numbers)
    maximum = max(numbers)
    return mean, minimum, maximum   # Python bundles these as a tuple

# Unpack the returned tuple into separate variables
data = [4, 7, 2, 9, 1, 8, 3]
mean, lo, hi = analyse(data)
print(f"Mean: {mean:.2f}, Min: {lo}, Max: {hi}")

# Or receive as a single tuple
result = analyse(data)
print(type(result))
print(result)

## 4.3 *args and **kwargs

**The problem they solve:**
Sometimes you want a function to accept a variable number of arguments — you don't know at design time how many the caller will provide.

- `*args` collects any extra **positional** arguments into a tuple
- `**kwargs` collects any extra **keyword** arguments into a dictionary

**Real-world example:** Python's built-in `print()` uses `*args` — that's how it accepts `print(a)`, `print(a, b)`, `print(a, b, c)`, etc.

In [ ]:
# *args — collect extra positional arguments as a tuple
def add_all(*numbers):
    """Sum any number of arguments."""
    print(f"  Received {len(numbers)} numbers: {numbers}")
    return sum(numbers)

print(add_all(1, 2))
print(add_all(1, 2, 3, 4, 5))
print(add_all(10, 20, 30, 40, 50, 60))

# You can unpack a list into *args using the * operator
values = [1, 2, 3, 4]
print(add_all(*values))   # same as add_all(1, 2, 3, 4)

In [ ]:
# **kwargs — collect extra keyword arguments as a dict
def create_profile(name, **details):
    """
    Build a user profile dict from keyword arguments.
    'name' is required; everything else is optional and variable.
    """
    profile = {"name": name}
    profile.update(details)   # merge in all the extra kwargs
    return profile

p1 = create_profile("Alice", age=25, city="London", job="Data Scientist")
p2 = create_profile("Bob", age=30, hobby="Photography")

print(p1)
print(p2)

# Combining all types: positional, *args, keyword defaults, **kwargs
def mixed(required, *args, separator=", ", **kwargs):
    parts = [str(required)] + [str(a) for a in args]
    print(separator.join(parts))
    print(kwargs)

mixed("hello", "world", "python", separator=" | ", author="Alice")

## 4.4 Scope — Local vs Global

**What is scope?**
Scope determines where in your code a variable is accessible. Python uses the LEGB rule to look up names:
1. **L**ocal — inside the current function
2. **E**nclosing — in an enclosing function (for nested functions)
3. **G**lobal — at the module level
4. **B**uilt-in — Python's built-in names like `len`, `print`

**Key rule:** Assigning a variable inside a function creates a LOCAL variable. This does NOT affect any global variable with the same name. This is intentional — functions should not have hidden side effects.

In [ ]:
x = 10   # global variable

def demo_scope():
    x = 99   # LOCAL variable — a completely separate x inside this function
    print(f"Inside function: x = {x}")    # 99

demo_scope()
print(f"Outside function: x = {x}")     # 10 — global unchanged!

# The global keyword — use sparingly, it makes code harder to reason about
counter = 0

def increment():
    global counter   # explicitly say "I mean the global counter"
    counter += 1

increment()
increment()
increment()
print(f"Counter: {counter}")   # 3

# BETTER approach: return values instead of modifying globals
def better_increment(count):
    return count + 1

counter = 0
counter = better_increment(counter)
print(f"Better counter: {counter}")

## 4.5 Lambda Functions

**What is a lambda?**
A lambda is a small, anonymous (unnamed) function defined in a single expression. Use them for simple, throwaway operations — especially when passing a function as an argument to another function.

**Syntax:** `lambda parameters: expression`

**When to use:** Short, single-use functions as arguments to `sorted()`, `map()`, `filter()`, `max()`, etc. For anything complex or reusable, write a regular `def` function instead.

In [ ]:
# Regular function vs lambda
def square(x):
    return x ** 2

square_lambda = lambda x: x ** 2   # equivalent one-liner

print(square(5))
print(square_lambda(5))

# Lambdas shine when used inline with higher-order functions
students = [
    {"name": "Bob",   "gpa": 3.2, "age": 22},
    {"name": "Alice", "gpa": 3.8, "age": 20},
    {"name": "Carol", "gpa": 3.5, "age": 21},
]

# Sort by GPA (descending) — lambda extracts the value to sort by
by_gpa = sorted(students, key=lambda s: s["gpa"], reverse=True)
for s in by_gpa:
    print(f"  {s['name']}: GPA {s['gpa']}")

# Sort by multiple criteria: first by GPA, then by age
by_gpa_then_age = sorted(students, key=lambda s: (-s["gpa"], s["age"]))

In [ ]:
# map() — apply a function to every element, returns a lazy iterator
numbers = [1, 2, 3, 4, 5, 6, 7, 8]

squares = list(map(lambda x: x**2, numbers))
print("Squares:", squares)

# filter() — keep only elements where function returns True
evens = list(filter(lambda x: x % 2 == 0, numbers))
print("Evens:", evens)

# These patterns are often replaced by list comprehensions (Module 09)
squares2 = [x**2 for x in numbers]         # same as map
evens2   = [x for x in numbers if x%2==0]  # same as filter
print("Comprehension squares:", squares2)

## 4.6 Decorators

**What is a decorator?**
A decorator is a function that takes another function and extends or modifies its behaviour, without changing the original function's source code.

Decorators use the `@decorator_name` syntax placed above the function definition.

**Real use cases:** Logging, timing, caching, authentication checks, input validation — any behaviour you want to add to multiple functions without repeating code.

In [ ]:
import time

# A decorator is a function that receives a function and returns a new function
def timer(func):
    """Decorator that prints how long the function takes to run."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)        # call the original function
        elapsed = time.time() - start
        print(f"  [{func.__name__}] took {elapsed:.4f}s")
        return result
    return wrapper

# Apply the decorator with @ syntax
@timer
def slow_sum(n):
    """Sum numbers 0 to n-1."""
    return sum(range(n))

@timer
def slow_sort(data):
    """Sort a list."""
    return sorted(data)

result = slow_sum(1_000_000)
print(f"  Result: {result}")

import random
data = random.sample(range(10000), 1000)
sorted_data = slow_sort(data)

# The @timer decorator works on ANY function — add it once, reuse everywhere

---

## Key Takeaways

- Functions should do ONE thing. Name them clearly (verb phrases: `calculate_tax`, `find_user`)
- Always write docstrings
- `*args` for variable positional args (tuple); `**kwargs` for variable keyword args (dict)
- Default arguments make common cases easy; keyword args make calls self-documenting
- Lambdas for short inline functions; `def` for anything reusable
- Decorators add cross-cutting behaviour without modifying original functions

## Exercises

[04_exercises.ipynb](exercises/04_exercises.ipynb) | [04_solutions.ipynb](exercises/04_solutions.ipynb)

## Next: [05 — OOP](../05_oop/05_oop.ipynb)
